# Social Media Interest Clustering for Marketing

## Objective: To identify distinct social sub-cultures among high school students using text-based interest keywords.

In [ ]:
#Comment out if not required
#%pip install pandas numpy matplotlib seaborn scikitlearn

## 1. Environment Setup and Data Loading

Unlike the previous datasets, this one is "high-dimensional" (many columns). We will focus on the interest keywords (from 'basketball' onwards) for clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Load the data
df = pd.read_csv('Dataset/Clustering_Marketing.csv')

# The interests start from 'basketball' (column index 4) to the end
interests = df.iloc[:, 4:]

print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

## 2. Preprocessing: Handling Missing Values and Scaling

Social media data often has missing values (especially in the 'age' column) and sparse data in interest keywords

In [ ]:
# Handle missing age values with the mean
df['age'] = df['age'].fillna(df['age'].mean())

# Interest data is often skewed; we standardize to give all interests equal weight
scaler = StandardScaler()
interests_scaled = scaler.fit_transform(interests)

## 3. Choosing the Number of Clusters (K)

For marketing segmentation, we typically look for 4–6 clusters to ensure the segments are large enough to be actionable but distinct enough to be useful.

In [ ]:
wcss = []
for k in range(2, 10):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(interests_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(range(2, 10), wcss, marker='o', linestyle='--')
plt.title('Elbow Method for Social Media Data')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.show()

## 4. Fitting the Model and 2D Visualization

Let's assume K=5 to capture diverse sub-cultures (e.g., athletes, "preppy" students, music lovers, etc.).

In [ ]:
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(interests_scaled)

# PCA for visualization
pca = PCA(n_components=2)
pca_data = pca.fit_transform(interests_scaled)
df_pca = pd.DataFrame(pca_data, columns=['PC1', 'PC2'])
df_pca['Cluster'] = df['Cluster']

plt.figure(figsize=(10, 7))
sns.scatterplot(x='PC1', y='PC2', hue='Cluster', data=df_pca, palette='tab10', alpha=0.6)
plt.title('Social Network Segments (PCA Projection)')
plt.show()